In [1]:
import numpy as np
import os
import random
import pandas as pd
from datasets import Dataset, DatasetDict, load_metric
import numpy as np
from huggingface_hub import login
import warnings
from transformers import BitsAndBytesConfig
import torch
from huggingface_hub.hf_api import HfFolder
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, TrainerCallback, Trainer
import transformers
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from peft import LoraConfig, get_peft_model, PeftModel



import json
import numpy as np
import pandas as pd
import random
from itertools import chain
from sentence_transformers import SentenceTransformer, util


random.seed(42)
np.random.seed(42)


train_data = np.load('train_data.npy', allow_pickle=True)
ice_data = np.load('ice_data.npy', allow_pickle=True)
test_data = np.load('test_data.npy', allow_pickle=True)

np.random.shuffle(train_data)
np.random.shuffle(ice_data)
np.random.shuffle(test_data)


model = SentenceTransformer('bert-base-nli-mean-tokens')

def extract_questions(data):
    return [item[0] for item in data]

train_text = extract_questions(train_data)
in_context_text = extract_questions(ice_data)

train_embeddings = model.encode(train_text, convert_to_tensor=True)
in_context_embeddings = model.encode(in_context_text, convert_to_tensor=True)

similarity_matrix = util.cos_sim(train_embeddings, in_context_embeddings)

combinations = [
    ("2-sim-dissim", 2, {"similar": 1, "dissimilar": 1, "random": 0}),
    ("2-half-random", 2, {"similar": 1, "dissimilar": 0, "random": 1}),
    ("2-random", 2, {"similar": 0, "dissimilar": 0, "random": 2}),
    ("3-sim-dissim", 3, {"similar": 2, "dissimilar": 1, "random": 0}),
    ("3-half-random", 3, {"similar": 2, "dissimilar": 0, "random": 1}),
    ("3-random", 3, {"similar": 0, "dissimilar": 0, "random": 3}),
    ("4-sim-dissim", 4, {"similar": 2, "dissimilar": 2, "random": 0}),
    ("4-half-random", 4, {"similar": 2, "dissimilar": 0, "random": 2}),
    ("4-random", 4, {"similar": 0, "dissimilar": 0, "random": 4}),
    ("5-sim-dissim", 5, {"similar": 3, "dissimilar": 2, "random": 0}),
    ("5-half-random", 5, {"similar": 3, "dissimilar": 0, "random": 2}),
    ("5-random", 5, {"similar": 0, "dissimilar": 0, "random": 5}),
]

num_training = len(train_text)
groups = [list(range(num_training))[i:i + 500] for i in range(0, num_training, 500)]

final_indices = []
combination_types = []


for group, (comb_type, num_examples, counts) in zip(groups, combinations):
    for idx in group:
        similarities = similarity_matrix[idx]
        sorted_indices = similarities.argsort(descending=True)
        most_similar = sorted_indices[:counts["similar"]].tolist()
        most_dissimilar = sorted_indices[-counts["dissimilar"]:].tolist() if counts["dissimilar"] > 0 else []
        random_indices = random.sample(range(len(in_context_text)), counts["random"]) if counts["random"] > 0 else []
        selected_indices = list(chain(most_similar, most_dissimilar, random_indices))
        random.shuffle(selected_indices)
        final_indices.append(selected_indices)
        combination_types.append(comb_type)

merged_list_train = list(zip(final_indices, combination_types))


2024-12-11 16:46:27.169525: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-11 16:46:27.212633: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-11 16:46:27.212665: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-11 16:46:27.213669: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-11 16:46:27.220748: I tensorflow/core/platform/cpu_feature_guar

In [2]:

# Function to process test data for a dataset
def process_test_data(test_data, in_context_data, similarity_matrix, combinations):
    num_test_examples = len(test_data)
    group_size = num_test_examples // len(combinations)  # 22 examples per combination
    print(group_size)
    groups = [list(range(num_test_examples))[i:i + group_size] for i in range(0, num_test_examples, group_size)]

    final_indices = []
    combination_types = []

    for group, (comb_type, num_examples, counts) in zip(groups, combinations):
        for idx in group:
            similarities = similarity_matrix[idx]
            sorted_indices = similarities.argsort(descending=True)
            most_similar = sorted_indices[:counts["similar"]].tolist()
            most_dissimilar = sorted_indices[-counts["dissimilar"]:].tolist() if counts["dissimilar"] > 0 else []
            random_indices = random.sample(range(len(in_context_data)), counts["random"]) if counts["random"] > 0 else []
            selected_indices = list(chain(most_similar, most_dissimilar, random_indices))
            random.shuffle(selected_indices)
            final_indices.append(selected_indices)
            combination_types.append(comb_type)

    return final_indices, combination_types

test_text = extract_questions(test_data)
test_embeddings = model.encode(test_text, convert_to_tensor=True)
similarity_matrix_test = util.cos_sim(test_embeddings, in_context_embeddings)

final_indices_test, combination_types_test = process_test_data(
    test_text, in_context_text, similarity_matrix_test, combinations
)

# Merged list for test data in both datasets
merged_list_test = list(zip(final_indices_test, combination_types_test))




200


In [3]:

import numpy as np

import warnings
warnings.filterwarnings('ignore')



base_template = """Instruction:
You are an expert assistant trained to jointly predict the toxicity and the gender for the given input from social media post and its response. 

Possible types of toxicity are: 'Toxic', and 'Non-Toxic'.  
Possible types of gender are: 'Male' and 'Female'.  

Output Format:
The output should be in the format: 'toxicity, gender'.

Examples:
"""

def put_example(data):
    
    template = """
{post}
Q: Predict the toxicity and the gender of the above post and response in the format toxicity, gender.
Answer: {tox}, {gend}
"""
    result_string = template.format(post=data[0], tox=data[1], gend=data[2])
    
    return result_string

def put_example_test(data):
    
    template = """
Now, solve for this example:
{post}
Q: Predict the toxicity and the gender of the above post and response in the format toxicity, gender.
Model Answer: {tox}, {gend}
"""
    result_string = template.format(post=data[0], tox=data[1], gend=data[2])
    
    return result_string

def put_example_test_for_test_data(data):
    template = """
Now, solve for this example:
{post}
Q: Predict the toxicity and the gender of the above post and response in the format toxicity, gender.
Model Answer: """
    result_string = template.format(post=data[0])
    
    return result_string





import ast
from datasets import Dataset, DatasetDict
def get_prompt(idx, data, Test=0):
    result = []
    temp = []
    target = []
    for index, id in enumerate(idx):
        temp_template = base_template
        indices_list = id[0]
        for i in range(len(indices_list)):
           example = put_example(ice_data[indices_list[i]])
           temp_template = temp_template + example
        
        if Test == 0:
            temp_template = temp_template + put_example_test(data[index])
        else:
            temp_template = temp_template + put_example_test_for_test_data(data[index])
        temp.append(temp_template)
        
        target.append((f'{data[index][1]}, {data[index][2]}'))
        result.append(f"{id[1]}")

    return {'input_text': temp, 'target_text': target, 'combination': result}
            
            
            
train_data = get_prompt(merged_list_train, train_data)
test_data = get_prompt(merged_list_test, test_data, Test=1)


train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)


#Convert DataFrames to Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Create DatasetDict
dataset_dict = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})


In [ ]:
dataset_dict

In [5]:
dataset_dict['train']['input_text'][0]

'Instruction:\nYou are an expert assistant trained to jointly predict the toxicity and the gender for the given input from social media post and its response. \n\nPossible types of toxicity are: \'Toxic\', and \'Non-Toxic\'.  \nPossible types of gender are: \'Male\' and \'Female\'.  \n\nOutput Format:\nThe output should be in the format: \'toxicity, gender\'.\n\nExamples:\n\nThe Muddled, Mixed Up Migrant Minister Maryam Monsef did not get shuffled out of cabinet.  She still has the big salary and other perks of cabinet.  But she was  demoted to the most useless portfolio in cabinet, the minster responsible for the status of women.  This boondoggle created by Trudeau 1.0 is even more redundant and patronizing now.  \n\nMonsef replaced Del Mastro as MP for Peterborough, giving the citizens a totally unqualified, utterly clueless chancer who pretended to be born in the wrong country because it made for a more electable backstory.\n\nAnd Del Mastro was fined, served 30 days in jail and was

In [4]:
from peft import LoraConfig, PeftConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub.hf_api import HfFolder
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

peft_config = LoraConfig(
    r= 8,          
    lora_alpha= 64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [5]:
HfFolder.save_token(os.environ.get("HF_TOKEN"))
model_id = "meta-llama/Llama-2-7b-chat-hf"

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map={"": "cuda:0"}, trust_remote_code=True,)

model.config.use_cache = True # silence the warnings
# model.config.pretraining_tp = 1
# model.gradient_checkpointing_enable()
# model.enable_input_require_grads()
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
#model.resize_token_embeddings(len(tokenizer))
#tokenizer.add_tokens(new_tokens)
#model.resize_token_embeddings(len(tokenizer))




tokenizer.padding_side = "left"
tokenizer.pad_token_id = tokenizer.bos_token_id
tokenizer.pad_token = tokenizer.bos_token
model.config.pad_token_id = tokenizer.bos_token_id
model = model.bfloat16()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [10]:
import torch
from tqdm import tqdm  # Import tqdm for progress bar

# Ensure you're in evaluation mode
model.eval()

# Define the batch size
batch_size = 4  # Adjust based on your GPU memory

def infer_batch(model, tokenizer, input_texts, batch_size, max_input_length):
    predictions = []
    num_batches = (len(input_texts) + batch_size - 1) // batch_size  # Calculate number of batches

    with torch.no_grad():
        for i in tqdm(range(num_batches), desc="Processing Batches", unit="batch"):
            start_idx = i * batch_size
            end_idx = min(start_idx + batch_size, len(input_texts))
            batch = input_texts[start_idx:end_idx]
            
            # Tokenize the batch
            inputs = tokenizer(batch, return_tensors="pt", padding="max_length", truncation=True, max_length=max_input_length)
            #print(inputs['input_ids'])
            input_ids = inputs["input_ids"].to(model.device)
            attention_mask = inputs["attention_mask"].to(model.device)
            
            # Forward pass
            outputs = model.generate(input_ids, attention_mask=attention_mask, max_new_tokens=16, num_return_sequences=1)
            #prob = model(inputs[0])
            #print(outputs.tolist())
            # Decode the predictions
            batch_predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            for pred in batch_predictions:
                if 'Model Answer:' in pred:
                    #print(pred)
                    _, result = pred.split('Model Answer:', 1)  # Split and get part after the delimiter
                    predictions.append(result.strip())
                else:
                    predictions.append(pred.strip())
    
    return predictions

# Test dataset
test_data = test_dataset['input_text']
# Get predictions
predictions = infer_batch(model, tokenizer, test_data, batch_size, max_input_length=1200)
#np.save('custom_loss_pred.npy', np.array(predictions))
# Optionally, compare with the true labels if needed
test_labels = test_dataset['target_text']


Processing Batches: 100%|██████████| 600/600 [1:29:30<00:00,  8.95s/batch]


In [11]:
predictions

['Toxicity: Non-Toxic, Gender: Female',
 'Toxicity: Non-Toxic\nGender: Female',
 'Toxicity: Toxic\nGender: Male',
 'Toxicity: Toxic\nGender: Male',
 'Instruction:\nYou are an expert assistant trained to jointly predict the toxicity and the gender for the given input from social media post and its response. \n\nPossible types of toxicity are: \'Toxic\', and \'Non-Toxic\'.  \nPossible types of gender are: \'Male\' and \'Female\'.  \n\nOutput Format:\nThe output should be in the format: \'toxicity, gender\'.\n\nExamples:\n\nLove is not a feeling. Love is not mere expression.\n\nLove is an act of the will, a sacrifice for the good of another.  The costlier & more cheerful the sacrifice the more we take on the image of Jesus.  Church teaching. \n\nIt\'s why we hang the simple compact image of the Crucifix in our home, office, bedroom: to help us to love as Jesus loved.\n\nThe love that man has for his wife should "converge" on the image of Jesus\'s love on the Cross.  And his wife should se

In [15]:

HfFolder.save_token(os.environ.get("HF_TOKEN"))
model_id = "meta-llama/Llama-2-7b-chat-hf"

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map={"": 0}, trust_remote_code=True,)

model.config.use_cache = True # silence the warnings
model.config.pretraining_tp = 1
model.gradient_checkpointing_enable()


tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


tokenizer2 = AutoTokenizer.from_pretrained(model_id, use_fast=True)
tokenizer2.pad_token = tokenizer.eos_token
tokenizer2.padding_side = "left"
#tokenizer.add_tokens(new_tokens)
#model.resize_token_embeddings(len(tokenizer))
model.enable_input_require_grads()


model = get_peft_model(model, peft_config).cuda()
#model = PeftModel.from_pretrained(model, 'fine_tuned_llama2_test').cuda()



def preprocess_function(example, tokenizer):
    #tokenizer.padding_side = "left"
    tokenizer.pad_token = tokenizer.bos_token
    model_inputs = tokenizer(example['input_text'], max_length=2048, padding='max_length', truncation=True)
    #tokenizer.padding_side = "right"
    tokenizer.pad_token = tokenizer.eos_token
    labels = tokenizer(example['target_text'], max_length=10, truncation=True, padding="max_length")["input_ids"]
    labels = [(label if label != tokenizer.pad_token_id else -100) for label in labels]
       
    
    
    model_inputs['labels'] = labels
    model_inputs['combination'] = example['combination']
    model_inputs['input_text'] = example['input_text']
    model_inputs['target_text'] = example['target_text']
    return model_inputs

def preprocess_dataset_individual(dataset):
    # Extract the examples
    examples = dataset.to_pandas().to_dict(orient='records')  # Convert to list of dicts

    # Process each example
    processed_examples = []
    for example in examples:
        processed_example = preprocess_function(example, tokenizer)
        processed_examples.append(processed_example)

    # Convert the processed examples back to a dataset
    processed_df = pd.DataFrame(processed_examples)
    return Dataset.from_pandas(processed_df)

# Process each split individually
processed_train_dataset = preprocess_dataset_individual(dataset_dict['train'])
processed_test_dataset = preprocess_dataset_individual(dataset_dict['test'])

processed_dataset_dict = DatasetDict({
    'train': processed_train_dataset,
    'test': processed_test_dataset
})



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [13]:
print(processed_train_dataset['input_ids'][0])

[1, 2799, 4080, 29901, 13, 3492, 526, 385, 17924, 20255, 16370, 304, 14002, 368, 8500, 278, 304, 29916, 24798, 322, 278, 23346, 363, 278, 2183, 1881, 515, 5264, 5745, 1400, 322, 967, 2933, 29889, 29871, 13, 13, 9135, 1687, 4072, 310, 304, 29916, 24798, 526, 29901, 525, 1762, 27375, 742, 322, 525, 12283, 29899, 1762, 27375, 4286, 259, 13, 9135, 1687, 4072, 310, 23346, 526, 29901, 525, 29924, 744, 29915, 322, 525, 29943, 331, 744, 4286, 259, 13, 13, 6466, 19191, 29901, 13, 1576, 1962, 881, 367, 297, 278, 3402, 29901, 525, 517, 29916, 24798, 29892, 23346, 4286, 13, 13, 1252, 9422, 29901, 13, 13, 1576, 341, 566, 29881, 839, 29892, 341, 11925, 5020, 341, 4481, 424, 7668, 6182, 314, 2598, 28604, 1258, 451, 679, 528, 3096, 839, 714, 310, 28966, 29889, 29871, 2296, 1603, 756, 278, 4802, 4497, 653, 322, 916, 639, 2039, 310, 28966, 29889, 29871, 1205, 1183, 471, 29871, 1261, 5715, 304, 278, 1556, 19315, 2011, 25648, 297, 28966, 29892, 278, 1375, 2475, 14040, 363, 278, 4660, 310, 5866, 29889, 298

In [16]:
import numpy as np

# Sample list of input IDs (replace this with your dataset)
input_ids_list = processed_test_dataset['input_ids']

# Define the padding token ID
padding_token_id = 1

# Calculate the effective length of each sequence (excluding padding tokens)
effective_lengths = [len([x for x in seq if x != padding_token_id]) for seq in input_ids_list]

# Compute optimal max length
optimal_max_length = {
    "max_length": max(effective_lengths),
    "95th_percentile_length": int(np.percentile(effective_lengths, 95)),
    "mean_length": int(np.mean(effective_lengths)),
}

print("Optimal max length analysis:", optimal_max_length)

Optimal max length analysis: {'max_length': 1691, '95th_percentile_length': 1172, 'mean_length': 725}


In [ ]:


training_args = TrainingArguments(
    output_dir = "./results_llama",
    num_train_epochs = 4,
    fp16 = False,
    bf16 = False,
    per_device_train_batch_size = 4,
    per_device_eval_batch_size = 4,
    #gradient_accumulation_steps = 25,
    gradient_checkpointing = True,
    max_grad_norm = 0.3,
    learning_rate = 2e-4,
    weight_decay = 0.001,
    optim = "paged_adamw_32bit",
    lr_scheduler_type = "cosine",
    max_steps = -1,
    warmup_ratio = 0.03,
    group_by_length = True,
    save_steps = 0,
    logging_steps = 20,
    logging_dir='./logs',
    report_to='none'
)

class CustomCallback(TrainerCallback):
    def __init__(self, model, tokenizer, test_dataset):
        self.model = model
        self.tokenizer = tokenizer
        self.test_dataset = test_dataset
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'

    def on_log(self, args, state, control, logs=None, **kwargs):
        if state.global_step % args.logging_steps == 0:
            if logs:
                print(f"Step: {state.global_step}")
                for key, value in logs.items():
                    print(f"{key}: {value}")
                
                # Select a random input text from the training data
                sample = random.choice(self.test_dataset)
                
                input_text = sample['input_text']
                # Tokenize and generate prediction
                inputs = self.tokenizer(input_text, return_tensors="pt", truncation=True, padding="max_length", max_length=1024).to(self.device)
                
                outputs = self.model.generate(**inputs, max_new_tokens=5)
                
                generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                start_idx = generated_text.find('Model Answer:') + len('Model Answer:')
                extracted_text = generated_text[start_idx:].strip()
                

                #print(f"Input text: {input_text}")
                #modified_generated_text = generated_text.replace(input_text, "").strip()
                print(f"Generated text: {extracted_text}")
                print(f"Target text: {sample['target_text']}\n")
                print('-------------------------------------------------')

from transformers import Trainer
custom_callback = CustomCallback(model, tokenizer2, processed_dataset_dict['test'])

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

template_ids = tokenizer.encode("Model Answer:", add_special_tokens=False)[2:]
collator = DataCollatorForCompletionOnlyLM(template_ids, tokenizer=tokenizer)




trainer = SFTTrainer(
    model=model,
    train_dataset=processed_dataset_dict['train'],
    eval_dataset=processed_dataset_dict['test'],  # Validation dataset
    peft_config=peft_config,
    #data_collator=collator,
    tokenizer=tokenizer,
    args=training_args,
    callbacks=[custom_callback]
)

trainer.train()


trainer.model.save_pretrained('./fine_tuned_llama2_rtgen')
tokenizer.save_pretrained('./fine_tuned_llama2_rtgen')

test_results = trainer.evaluate(eval_dataset=processed_dataset_dict['test'])
print(f"Test Loss: {test_results['eval_loss']}")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
20,7.299700
40,4.500500
60,0.939600
80,0.729600
100,0.678500


Step: 20
loss: 7.2997
grad_norm: 2.785341739654541
learning_rate: 4.4444444444444447e-05
epoch: 0.02666666666666667
Generated text: 
Target text: Mixed, Male

-------------------------------------------------
Step: 40
loss: 4.5005
grad_norm: 3.0893661975860596
learning_rate: 8.888888888888889e-05
epoch: 0.05333333333333334
Generated text: nobody Hinweis
Target text: Neutral, Male

-------------------------------------------------
Step: 60
loss: 0.9396
grad_norm: 0.4266405701637268
learning_rate: 0.00013333333333333334
epoch: 0.08
Generated text: Posϊ sierp
Target text: Mixed, Male

-------------------------------------------------
Step: 80
loss: 0.7296
grad_norm: 0.28407707810401917
learning_rate: 0.00017777777777777779
epoch: 0.10666666666666667
Generated text: Neigh Hinweis
Target text: Negative, Male

-------------------------------------------------
Step: 100
loss: 0.6785
grad_norm: 0.2845520079135895
learning_rate: 0.00019999417253661235
epoch: 0.13333333333333333
Generated text: 